# 2.1 Train pipeline — cashback + loyalty connection + dynamic FS


In [ ]:
# PROJECT_ROOT — ветка dynamic, в том числе при запуске Jupyter из общего каталога.
from pathlib import Path
import sys, os
_branch = 'dynamic'
_cwd = Path.cwd().resolve()
_candidates = [_cwd, _cwd / _branch, _cwd.parent / _branch]
PROJECT_ROOT = next((x for x in _candidates if x.name == _branch and (x / 'srcc' / 'nbc_project').is_dir()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError(f'Не найдена папка {_branch}/srcc/nbc_project; запустите из ветки или её родителя')
SRC_PATH = PROJECT_ROOT / 'srcc'
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))
os.chdir(PROJECT_ROOT)
print('PROJECT_ROOT:', PROJECT_ROOT)
EXPERIMENT_NAME = '202610_dynamic'
RUN_TAG = '202610_dynamic'


In [ ]:
import pandas as pd

from nbc_project.train_pipeline_workflow import (
    train_cashback_connected_clean,
    train_loyalty_connection_clean,
)
from nbc_project.training_visuals import (
    make_regression_report,
    make_classification_report,
    plot_rolling_backtest_summary,
)
from nbc_project.rolling_backtest import run_rolling_backtest
from nbc_project.feature_lists import get_feature_list

import gc

In [ ]:
!mkdir -p data/datasets

In [ ]:
# Настройки октябрьских источников для Dynamic FS.
DATASET_POSTFIX = 'nbc_monthly_202610'
VALID_MONTH = '2026-10-01'
INFERENCE_MONTH = '2026-12-01'


In [ ]:
# Копируем именно октябрьские витрины, не используем старые july/sept.
from pathlib import Path
import subprocess, shutil
LOCAL_DATASETS = PROJECT_ROOT / 'data' / 'datasets'
LOCAL_DATASETS.mkdir(parents=True, exist_ok=True)

def copy_table_from_hdfs(table_name):
    table = table_name.split('.')[-1]
    dst = LOCAL_DATASETS / table
    roots = ('/warehouse/tablespace/managed/hive/sandbox_crft.db',
             '/warehouse/tablespace/external/hive/sandbox_crft.db')
    sources = [f'{root}/{table}' for root in roots]
    src = next((s for s in sources if subprocess.run(['hdfs','dfs','-test','-d',s]).returncode == 0), None)
    if src is None:
        raise FileNotFoundError(f'HDFS таблица {table_name} не найдена в managed/external: {sources}')
    if dst.exists():
        shutil.rmtree(dst)
    subprocess.run(['hdfs', 'dfs', '-copyToLocal', src, str(LOCAL_DATASETS)], check=True)
    assert dst.is_dir() and any(dst.rglob('*.parq*')), f'Нет Parquet файлов в {dst}'
    print(f'{table_name}: {src} -> {dst}')

copy_table_from_hdfs(f'sandbox_crft.cb_train_modular_full_connected_{DATASET_POSTFIX}')
copy_table_from_hdfs(f'sandbox_crft.cb_sample_for_binary_modular_full_{DATASET_POSTFIX}')


In [ ]:
DATA_CASHBACK_FULL = (
    "data/datasets/cb_train_modular_full_connected_nbc_monthly_202610"
)
DATA_LOYAL_FLG_FULL = (
    "data/datasets/cb_sample_for_binary_modular_full_nbc_monthly_202610"
)


USE_DYNAMIC_FS = True

# Финальная конфигурация после benchmark и переноса выбранных подходов
# на модели, для которых отдельный benchmark не прогоняли.
FS_BY_MODEL = {'cashback_connected': 'psi_importance_auto_n', 'loyalty_connect': 'psi_importance_auto_n'}

DYNAMIC_FS_PRESETS = {
    "psi_importance_auto_n": {
        "method": "psi_importance",
        "psi_threshold": 0.20,
        "psi_reference_months": 3,
        "auto_top_n": True,
        "top_n_candidates": [100, 150, 200, 250, 300],
        "sample_rows": 150_000,
        "selector_iterations": 200,
        "report_dir": "reports/feature_selection",
    },
    "psi_rfe_loss_auto_n": {
        "method": "psi_rfe",
        "psi_threshold": 0.20,
        "psi_reference_months": 3,
        "rfe_algorithm": "loss_function_change",
        "rfe_steps": 5,
        "auto_top_n": True,
        "top_n_candidates": [100, 150, 200, 250, 300],
        "sample_rows": 150_000,
        "selector_iterations": 180,
        "report_dir": "reports/feature_selection",
    },
}

CORRELATION_PREFILTER = {
    "drop_correlated_features": True,
    "correlation_threshold": 0.95,
    "correlation_method": "pearson",
    "correlation_max_rows": 20_000,
    "correlation_min_unique_values": 20,
}

for _cfg in DYNAMIC_FS_PRESETS.values():
    _cfg.update(CORRELATION_PREFILTER)

PROTECTED_OFFER_FEATURES = [
    c
    for c in get_feature_list("data_fm_must_have_features")
    if c.startswith("offered_percent_cat_")
]

if not PROTECTED_OFFER_FEATURES:
    PROTECTED_OFFER_FEATURES = [
        c
        for c in get_feature_list("must_have_offered_percent_features")
        if c.startswith("offered_percent_cat_")
    ]

def make_fs_config(model_key):
    preset = FS_BY_MODEL[model_key]
    cfg = DYNAMIC_FS_PRESETS[preset].copy()
    cfg["protected_features"] = PROTECTED_OFFER_FEATURES.copy()
    return cfg

print("FINAL FS_BY_MODEL:", FS_BY_MODEL)
print("protected offered_percent:", len(PROTECTED_OFFER_FEATURES))


def fs_for(model_key):
    return make_fs_config(model_key)

RUN_ROLLING_BACKTEST = False


## Параметры периода

In [ ]:
TRAIN_START='2025-09-01'
TRAIN_END_EXCLUSIVE='2026-10-01'
VALID_MONTH='2026-10-01'
TEST_MONTH=None
MODEL_DIR=str(PROJECT_ROOT / 'experiment_outputs' / '202610_dynamic' / 'models' / 'train_pipeline')
Path(MODEL_DIR).mkdir(parents=True, exist_ok=True)
print('split:', TRAIN_START, TRAIN_END_EXCLUSIVE, VALID_MONTH, TEST_MONTH)


## 1. Регрессия cashback для подключённых клиентов

In [ ]:
from pathlib import Path
assert Path(DATA_CASHBACK_FULL if USE_DYNAMIC_FS else DATA_CASHBACK).is_dir(), 'Не найдена октябрьская cashback-выборка'
assert TEST_MONTH is None and VALID_MONTH == '2026-10-01'
cashback_result = train_cashback_connected_clean(
    data_path=DATA_CASHBACK_FULL if USE_DYNAMIC_FS else DATA_CASHBACK,
    train_start=TRAIN_START, train_end_exclusive=TRAIN_END_EXCLUSIVE, valid_month=VALID_MONTH, test_month=TEST_MONTH,
    model_dir=MODEL_DIR, feature_selection=fs_for('cashback_connected'),
    model_key='cashback_connected', model_filename='cashback_connected.cbm',
)
print('cashback features:',len(cashback_result['feature_cols']))
cashback_result['metrics']


In [ ]:
split_targets = {
    "train": cashback_result["train"]["target_cashback_month"],
    "valid": cashback_result["valid"]["target_cashback_month"],
}

if not cashback_result["test"].empty:
    split_targets["test"] = cashback_result["test"]["target_cashback_month"]
    y_true_cashback = cashback_result["test"]["target_cashback_month"]
    y_pred_cashback = cashback_result["pred_test"]
    cashback_report_suffix = 'test_oot'
else:
    y_true_cashback = cashback_result["valid"]["target_cashback_month"]
    y_pred_cashback = cashback_result["pred_valid"]
    cashback_report_suffix = 'valid'

cashback_visuals = make_regression_report(
    model_key=f"cashback_connected_{cashback_report_suffix}",
    target_col="target_cashback_month",
    y_true=y_true_cashback,
    y_pred=y_pred_cashback,
    output_dir="reports/training_visuals",
    split_targets=split_targets,
    model=cashback_result["model"],
)


In [ ]:
cashback_visuals

In [ ]:
if RUN_ROLLING_BACKTEST:
    cashback_rolling = run_rolling_backtest(
        df=pd.concat([cashback_result['train'], cashback_result['valid']], ignore_index=True),
        target_col='target_cashback_month',
        task_type='regression',
        model_key='cashback_connected',
        output_dir='reports/rolling_backtest',
        month_col='forecast_month',
        ignore_cols=cashback_result['config'].ignore_cols,
        feature_cols=cashback_result['feature_cols'],
        min_train_months=3,
        model_params=ROLLING_MODEL_PARAMS,
    )

    # plot_rolling_backtest_summary('cashback_connected', rolling_dir='reports/rolling_backtest', output_dir='reports/training_visuals')

In [ ]:
del cashback_result, cashback_visuals
gc.collect()

### Dynamic FS для loyalty: отдельный calibration holdout

При `USE_DYNAMIC_FS=True` feature selection использует только `train + valid_model`. `valid_calib` не участвует в PSI / RFE / permutation / null importance и остаётся отдельной половиной месяца для isotonic calibration и проверки качества.


## 2. Бинарная модель подключения loyalty

In [ ]:
loyalty_result = train_loyalty_connection_clean(
    data_path=DATA_LOYAL_FLG_FULL if USE_DYNAMIC_FS else DATA_LOYAL_FLG,
    train_start=TRAIN_START, train_end_exclusive=TRAIN_END_EXCLUSIVE, valid_month=VALID_MONTH, test_month=TEST_MONTH,
    model_dir=MODEL_DIR, feature_selection=fs_for('loyalty_connect'),
    model_key='loyalty_connect', model_filename='loyalty_connect.cbm', calibrator_filename='isotonic_calibrator_loyalty.pkl',
)
print('loyalty features:',len(loyalty_result['feature_cols']))
loyalty_result['metrics']


In [ ]:
if not loyalty_result['test'].empty:
    y_true = loyalty_result['test']['target_loyalty_connected_flg']
    y_score = loyalty_result['pred_test']
    report_suffix = 'test'
else:
    y_true = loyalty_result['valid_calib']['target_loyalty_connected_flg']
    y_score = loyalty_result['pred_valid_calib']
    report_suffix = 'valid_calib'

split_targets = {
    'train': loyalty_result['train']['target_loyalty_connected_flg'],
    'valid_model': loyalty_result['valid_model']['target_loyalty_connected_flg'],
    'valid_calib': loyalty_result['valid_calib']['target_loyalty_connected_flg'],
}

if not loyalty_result['test'].empty:
    split_targets['test'] = loyalty_result['test']['target_loyalty_connected_flg']

loyalty_visuals = make_classification_report(
    model_key=f'loyalty_connect_{report_suffix}',
    target_col='target_loyalty_connected_flg',
    y_true=y_true,
    y_score=y_score,
    output_dir='reports/training_visuals',
    split_targets=split_targets,
    model=loyalty_result['model'],
    threshold=0.5,
)

In [ ]:
loyalty_visuals

In [ ]:
if RUN_ROLLING_BACKTEST:
    loyalty_rolling = run_rolling_backtest(
        df=pd.concat([
            loyalty_result['train'],
            loyalty_result['valid_model'],
            loyalty_result['valid_calib'],
        ], ignore_index=True),
        target_col='target_loyalty_connected_flg',
        task_type='classification',
        model_key='loyalty_connect',
        output_dir='reports/rolling_backtest',
        month_col='forecast_month',
        ignore_cols=loyalty_result['config'].ignore_cols,
        feature_cols=loyalty_result['feature_cols'],
        min_train_months=3,
        model_params=ROLLING_MODEL_PARAMS,
    )

    # plot_rolling_backtest_summary('loyalty_connect', rolling_dir='reports/rolling_backtest', output_dir='reports/training_visuals')

In [ ]:
del loyalty_result
gc.collect()

In [ ]:
import numpy as np
import pandas as pd
from catboost import Pool

def build_loyalty_shap_shift(
    loyalty_result,
    sample_size=30000,
    random_state=42,
):
    model = loyalty_result["model"]

    valid_calib = loyalty_result["valid_calib"].copy()
    test = loyalty_result["test"].copy()

    if test.empty:
        raise ValueError("TEST/OOT пустой")

    features = list(model.feature_names_)
    cat_idx = list(model.get_cat_feature_indices())

    X_valid = (
        valid_calib
        .reindex(columns=features)
        .copy()
    )

    X_test = (
        test
        .reindex(columns=features)
        .copy()
    )

    if len(X_valid) > sample_size:
        X_valid = X_valid.sample(
            sample_size,
            random_state=random_state,
        )

    if len(X_test) > sample_size:
        X_test = X_test.sample(
            sample_size,
            random_state=random_state,
        )

    pool_valid = Pool(
        X_valid,
        cat_features=cat_idx,
    )

    pool_test = Pool(
        X_test,
        cat_features=cat_idx,
    )

    shap_valid = model.get_feature_importance(
        pool_valid,
        type="ShapValues",
    )

    shap_test = model.get_feature_importance(
        pool_test,
        type="ShapValues",
    )

    sv_valid = shap_valid[:, :-1]
    sv_test = shap_test[:, :-1]

    importance = model.get_feature_importance()

    report = pd.DataFrame({
        "feature": features,
        "importance": importance,

        "shap_valid_mean":
            sv_valid.mean(axis=0),

        "shap_test_mean":
            sv_test.mean(axis=0),

        "shap_shift":
            sv_test.mean(axis=0)
            - sv_valid.mean(axis=0),

        "abs_shap_valid":
            np.abs(sv_valid).mean(axis=0),

        "abs_shap_test":
            np.abs(sv_test).mean(axis=0),
    })

    report["abs_shap_shift"] = (
        report["shap_shift"].abs()
    )

    report["in_valid_mean"] = [
        X_valid[c].mean()
        if pd.api.types.is_numeric_dtype(X_valid[c])
        else np.nan
        for c in features
    ]

    report["in_test_mean"] = [
        X_test[c].mean()
        if pd.api.types.is_numeric_dtype(X_test[c])
        else np.nan
        for c in features
    ]

    report["feature_mean_shift"] = (
        report["in_test_mean"]
        - report["in_valid_mean"]
    )

    report = report.sort_values(
        "abs_shap_shift",
        ascending=False,
    ).reset_index(drop=True)

    return report


dynamic_shift = build_loyalty_shap_shift(
    loyalty_result,
    sample_size=30000,
)

display(dynamic_shift.head(50))

dynamic_shift.to_csv(
    "dynamic_shift.csv",
    index=False,
)

print(
    "saved:",
    "dynamic_shift.csv",
    "rows:",
    len(dynamic_shift),
)